# Delays, pipelines and memory {#sec-delays}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c04-delays.ipynb)

A referral does not join the waiting list the moment a GP writes it, and a GP does not learn the true waiting time the moment it changes. Patients move through the list band by band, from under three months to over a year. Each of these is a **delay**: a gap in time between a cause and its effect. Delays decide how fast a system can respond, how far it overshoots and how wrong its decision makers can be. This chapter builds the delays of the waiting list and measures them from the data.

The figure after the setup cell shows the list as four waiting-time bands, the structure we fit to the data in section 3.

**Who it is for.** Modellers who need to represent pipelines, queues, perception or forecasts, and analysts who want to estimate waiting times from counts. **Who can skip.** If you know the difference between material and information delays, can derive Little's law and have built coflows before, read sections 3, 4 and 6 for the running case and skip the rest. **Prerequisites.** Chapter 3, Loops that grow, settle and tip, in particular the Residence Time and Close Gap molecules. **Time.** About three and a half hours, with the exercises.

The data are synthetic, shaped like published Irish waiting-list counts; chapter 10, Data, calibration and uncertainty, explains how they were made.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 4
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])

# Okabe-Ito colours with a different line style each, readable in greyscale
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
             "#E69F00", "#56B4E9", "#000000", "#F0E442"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)),
          (0, (1, 3)), (0, (8, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__, "| data:", wl.shape[0], "months,",
      wl.month.min().strftime("%b %Y"), "to", wl.month.max().strftime("%b %Y"))

In [ ]:
def stock(i, name, init, units, non_negative=True):
    """A stock for a model spec."""
    return {"id": i, "name": name, "init": str(init), "units": units,
            "nonNegative": non_negative, "x": 0, "y": 0}


def flow(i, name, eq, units, src=None, dst=None):
    """A flow for a model spec, from stock src to stock dst."""
    f = {"id": i, "name": name, "eq": eq, "units": units, "x": 0, "y": 0}
    if src:
        f["from"] = src
    if dst:
        f["to"] = dst
    return f


def var(i, name, eq, units):
    """A variable (a constant or an equation) for a model spec."""
    return {"id": i, "name": name, "eq": str(eq), "units": units,
            "x": 0, "y": 0}

In [ ]:
#| fig-alt: Four boxes in a row for the waiting-time bands, linked by ageing arrows; referrals enter the first box and treatments leave every box.
from matplotlib.patches import FancyArrowPatch, Rectangle


def draw_bands(ax):
    """The waiting list as four bands, each ageing into the next."""
    names = ["0 to 3\nmonths", "3 to 6\nmonths", "6 to 12\nmonths",
             "12 months\nand over"]
    pipe = dict(arrowstyle="-|>", mutation_scale=16, lw=3.5, color="0.35")
    xs = [0.10, 0.33, 0.56, 0.79]
    for x, n in zip(xs, names):
        ax.add_patch(Rectangle((x, 0.45), 0.15, 0.25, fill=False, lw=2))
        ax.text(x + 0.075, 0.575, n, ha="center", va="center", fontsize=9)
        ax.add_patch(FancyArrowPatch((x + 0.075, 0.45), (x + 0.075, 0.12),
                                     **pipe))
    for a, b in zip(xs[:-1], xs[1:]):
        ax.add_patch(FancyArrowPatch((a + 0.15, 0.575), (b, 0.575), **pipe))
    ax.add_patch(FancyArrowPatch((-0.05, 0.575), (0.10, 0.575), **pipe))
    ax.text(0.02, 0.65, "Referrals", ha="center", fontsize=9)
    ax.text(0.5, 0.03, "Treatments from every band (shorter waits "
            "weighted up)", ha="center", fontsize=9)
    ax.text(0.5, 0.85, "ageing: each band drains into the next",
            ha="center", fontsize=9)
    ax.set_xlim(-0.06, 1.0)
    ax.set_ylim(0, 0.95)
    ax.axis("off")


fig, ax = plt.subplots(figsize=(7, 3))
draw_bands(ax)
ax.set_title("The waiting list as an ageing chain")
plt.show()

Figure: The waiting list as an ageing chain. Referrals enter the youngest band; patients who are not treated age into the next band; treatments come out of every band.

**Learning outcomes.** By the end of the chapter you can:

1. **distinguish** material from information delays and **choose** the order of a delay to match the spread you expect (Try it 1);
2. **build** a staged pipeline and **estimate** how much sits in it and how long a change takes to pass through (Try it 2);
3. **fit** an ageing chain to banded counts and **interpret** what the fitted priority says (Try it 3);
4. **compute** an average waiting time from counts with Little's law and **state** when it holds (Try it 4);
5. **model** perception and forecasting as information delays and **diagnose** their effect on decisions (Try it 5 to 7);
6. **track** an attribute such as time waited with a coflow (Try it 8).

## 1. Every delay hides a stock

A **material delay** moves things that are conserved: parcels, patients, letters. While the output lags the input, the difference has to sit somewhere, and that somewhere is a stock. The **Material Delay** molecule (`material`), after Hines, *Molecules of Structure* [@hines_molecules], sends a flow through three stages, each draining in proportion to its contents:

$$
\text{Flow out of stage } i = \frac{\text{In transit}_i}{D/3}, \qquad \text{In transit}^{*} = \text{Inflow} \times D .
$$

Our version fixes the number of stages at three and names the total `Transit delay`, $D$.

In [ ]:
#| fig-alt: A flat line of 8 sent per week and an S-shaped arrivals curve that starts at zero and reaches 8 by about week 25.
mat = lb.molecule("material")
print(mat.equations())
pipe = mat.run()
stages = ["In transit 1", "In transit 2", "In transit 3"]
pipe["In transit"] = pipe[stages].sum(axis=1)
print(f"\nin transit at week 60: {pipe['In transit'].iloc[-1]:.1f} units "
      f"(8 a week x 9 weeks = {8 * 9})")
print(f"arrivals in week 9: {pipe.loc[9.0, 'Arrivals']:.2f} a week")

fig, ax = plt.subplots()
pipe["Entering"].plot(ax=ax, label="sent")
pipe["Arrivals"].plot(ax=ax, label="arriving")
ax.set(title="Material Delay: a 9-week, three-stage pipeline",
       xlabel="time (weeks)", ylabel="flow (units/week)")
ax.legend()
plt.show()

Figure: Material Delay. Units are sent at 8 a week from the start; arrivals build up slowly as the pipeline fills.

The pipeline fills to 72.0 units, exactly 8 a week times 9 weeks. One full delay after the first shipment, arrivals are still only 4.64 a week. Anyone who judges a supplier by arrivals alone will think it is failing for weeks. When the container ship *Ever Given* blocked the Suez Canal for six days in March 2021, the effect on European ports came weeks later, and lasted far longer than the blockage, because hundreds of ships already in the pipeline were held up behind it.

**The order of a delay.** Splitting a delay into $n$ stages changes the *shape* of the output, not its average. The **Cascaded Levels** molecule (`cascade`), after Hines, *Molecules of Structure* [@hines_molecules], chains three stages; a first-order delay is the Residence Time molecule from chapter 3. Ours starts with a batch of 60 in the first stage and no inflow, so its default run shows a pulse moving through. The helper below builds a chain of any order, drops in a batch of 100 and watches it come out.

In [ ]:
#| fig-alt: "Exit rates of a batch through a 12-month delay of order 1, 3 and 12: highest at once for order 1, a hump near month 8 for order 3, a narrow peak near month 11 for order 12."
def delay_chain(n, delay, batch=100.0):
    """A batch dropped into an n-stage chain with a total delay time."""
    st = [stock(f"s{i}", f"Stage {i}", batch if i == 1 else 0, "units")
          for i in range(1, n + 1)]
    fl = [flow(f"f{i}", f"Out of {i}", f"Stage_{i} / Stage_time",
               "units/month", src=f"s{i}",
               dst=f"s{i + 1}" if i < n else None)
          for i in range(1, n + 1)]
    return lb.Model.from_spec({
        "name": f"Delay of order {n}", "timeUnit": "month", "T": 36,
        "dt": 0.05, "stocks": st, "flows": fl,
        "vars": [var("st", "Stage time", delay / n, "months")]})


fig, ax = plt.subplots()
for n in (1, 3, 12):
    d = delay_chain(n, 12).run()
    exits = d[f"Out of {n}"]
    exits.plot(ax=ax, label=f"order {n}")
    left = d[[f"Stage {i}" for i in range(1, n + 1)]].sum(axis=1)
    print(f"order {n:>2}: exits peak in month {exits.idxmax():5.2f}; "
          f"{100 - left.loc[6.0]:5.1f}% out by month 6, "
          f"{100 - left.loc[24.0]:5.1f}% by month 24")
ax.set(title="One batch of 100 through a 12-month delay",
       xlabel="time (months)", ylabel="exits (units/month)")
ax.legend()
plt.show()
cas = lb.molecule("cascade").run()           # 60 units, 3 stages of 4
print(f"Cascaded Levels: outflow peaks in month {cas['Outflow'].idxmax()}")

Figure: The same 12-month delay with 1, 3 and 12 stages. More stages mean fewer early exits and a thinner tail.

The first-order delay releases 39.4% of the batch in the first 6 months; the twelfth-order one only 1.7%. By month 24 the order is reversed: 86.5% against 99.8%. Exits from the third-order delay peak in month 7.95, and the Cascaded Levels molecule (three stages of 4 months) peaks in month 7.9. A first-order delay suits a well-mixed pool where anyone may leave next, such as patients on a list treated in random order. Higher orders suit processes with steps that must happen in sequence; an infinite order is a fixed pipeline, where everything takes exactly $D$.

**Try it 1.**

- *Check.* For an $n$-stage delay the exit rate peaks at $(n-1)/n \times D$. Confirm it for $n = 3$ and $D = 12$.
- *Apply.* Double the delay to 24 months for order 3. When do exits peak?
- *Extend.* Compute the mean exit time for orders 1, 3 and 12. What does that tell you about choosing an order?

<details><summary>Worked answer</summary>

The rule gives 8.0 months; the run peaks in month 7.95 (the step is 0.05 months). With 24 months, exits peak in month 15.95, again two-thirds of the delay. The mean exit time is 11.94 to 11.95 months for all three orders: the average is the delay time whatever the order. Choose the order from the *spread* you see in the real process, not from the average.
</details>

In [ ]:
d = delay_chain(3, 12).run()
print("order 3, 12 months: exits peak in month",
      d["Out of 3"].idxmax(), "| rule (n - 1) / n x delay =", 2 / 3 * 12)
d = delay_chain(3, 24).run(T=72)
print("order 3, 24 months: exits peak in month", d["Out of 3"].idxmax())
for n in (1, 3, 12):
    out = delay_chain(n, 12).run(T=120)[f"Out of {n}"]
    mean = (out.index * out).sum() * 0.05 / 100
    print(f"order {n:>2}: mean exit time {mean:.2f} months")

## 2. The referral backlog in stages

In the running case, a GP's referral letter is first triaged by a consultant, then booked onto the list. Suppose each step takes 0.75 months on average. The two steps form a second-order material delay between `GP referrals` and `Referrals` to the list. At month 6, GPs start referring 10% more (400 a month).

In [ ]:
#| fig-alt: A step in GP referrals from 4,000 to 4,400 at month 6, and referrals to the list following smoothly over the next few months.
BACKLOG = {
    "name": "Referral backlog", "timeUnit": "month", "T": 36, "dt": 0.05,
    "stocks": [stock("tri", "Awaiting triage", 3000, "referrals"),
               stock("bok", "Awaiting booking", 3000, "referrals")],
    "flows": [
        flow("gp", "GP referrals", "Base_referrals + STEP(400, 6)",
             "referrals/month", dst="tri"),
        flow("tr", "Triaged", "Awaiting_triage / Stage_time",
             "referrals/month", src="tri", dst="bok"),
        flow("ref", "Referrals", "Awaiting_booking / Stage_time",
             "patients/month", src="bok")],
    "vars": [var("base", "Base referrals", 4000, "referrals/month"),
             var("st", "Stage time", 0.75, "months")]}
backlog = lb.Model.from_spec(BACKLOG)
b = backlog.run()
b["Backlog"] = b["Awaiting triage"] + b["Awaiting booking"]
print(b.loc[[0.0, 6.0, 7.0, 8.0, 12.0, 36.0],
            ["GP referrals", "Referrals", "Backlog"]].round(0))
late = b["Referrals"][b.index > 6]
print("referrals to the list reach 95% of the step in month",
      late.index[late >= 4000 + 0.95 * 400][0])

fig, ax = plt.subplots()
b[["GP referrals", "Referrals"]].loc[:18].plot(ax=ax)
ax.set(title="A 10% step in GP referrals reaches the list late",
       xlabel="time (months)", ylabel="flow (referrals/month)")
ax.legend(["sent by GPs", "joining the list"])
plt.show()

Figure: The referral backlog as a two-stage pipeline. The step sent by GPs (solid) reaches the list (dashed) only gradually.

Before the step, 6,000 referrals sit in the backlog: 4,000 a month times 1.5 months. After it, the backlog grows to 6,600, and referrals reaching the list hit 95% of the step only in month 9.5, three and a half months after GPs changed. For those months, a manager counting additions to the list would underestimate demand.

**Try it 2.**

- *Check.* Without running it, how many referrals sit in the backlog in the new steady state if each stage takes 1.5 months?
- *Apply.* Run it. When do 95% of the extra referrals reach the list?
- *Extend.* Why does a longer backlog make a growing list look healthier than it is?

<details><summary>Worked answer</summary>

4,400 a month times 3 months gives 13,200, as the run shows. Ninety-five per cent of the step arrives in month 13.3, more than seven months after it began. The backlog is a hidden waiting list: patients in it are waiting, but are not counted. A longer backlog hides more of them, so any growth in demand shows up on the list later and smaller.
</details>

In [ ]:
for st in (0.75, 1.5):
    x = backlog.run(stage_time=st)
    late = x["Referrals"][x.index > 6]
    end = x["Awaiting triage"].iloc[-1] + x["Awaiting booking"].iloc[-1]
    print(f"stage time {st} months: backlog at month 36 = {end:,.0f}; "
          f"95% of the step reaches the list in month "
          f"{late.index[late >= 4380][0]}")

## 3. Ageing chains: the waiting-time bands

An **ageing chain** splits one stock into stages that items pass through in order, each stage a first-order delay. Unlike a plain delay, a chain lets you see and act on each stage. The **Aging Chain** molecule (`aging`), after Hines, *Molecules of Structure* [@hines_molecules], is a population in three age groups:

$$
\frac{dS_i}{dt} = \frac{S_{i-1}}{\tau_{i-1}} - \frac{S_i}{\tau_i}, \qquad S_0/\tau_0 = \text{Births}.
$$

The **Aging Chain with PDY** molecule (`agingPDY`), after Hines, *Molecules of Structure* [@hines_molecules], carries the same chain for staff (rookies, experienced, veterans), with a different productivity in each stage. Our versions start in equilibrium, so any movement you see is caused by the change you make.

In [ ]:
ag = lb.molecule("aging")
dec = ag.run(preset="Birth decline")
for s in ("Youth", "Working age", "Elderly"):
    first = dec.index[dec[s] < 0.9 * dec[s].iloc[0]][0]
    print(f"{s:<12}: {dec[s].iloc[0]:4.0f} at start, 10% down in year "
          f"{first}")

pdy = lb.molecule("agingPDY")
for h in (20, 40):
    r = pdy.run(hiring=h)
    staff = r[["Rookies", "Experienced", "Veterans"]].sum(axis=1)
    per = r["Production"] / staff
    print(f"hiring {h}/year: staff {staff.iloc[0]:.0f} -> "
          f"{staff.iloc[-1]:.0f}, output per person {per.iloc[0]:.3f} -> "
          f"lowest {per.min():.3f} in year {per.idxmin()}")

When births fall from 20 to 8 a year, the youth group is 10% down after 3.3 years, the working-age group only after 21.4 years and the elderly after 33.6. Each stage passes the shock on later and more gently. South Korea, where births have fallen steeply since the 1970s, has seen it in this order: first schools closing, then a shrinking workforce decades later. In the staff chain, doubling hiring grows the firm from 720 to 1,365 people, but output per person dips from 0.914 to 0.894 in year 5.2, because so many newcomers are still rookies.

Now the running case. The data give the list in four bands: under 3 months, 3 to 6, 6 to 12 and 12 months or more. That is an ageing chain, with ageing times fixed by the band widths (3, 3 and 6 months). Treatments come out of every band. We let the youngest band be treated a little more often than its size alone suggests (urgent cases), through a `Priority weight`:

$$
\text{Treated}_i = \text{Treatments} \times \frac{w_i\,B_i}{\sum_j w_j B_j}, \qquad w_1 = \text{Priority weight},\ w_2 = w_3 = w_4 = 1 .
$$

Referrals start at 4,000 a month and grow by the fraction 0.0008 a month fitted in chapter 3, Loops that grow, settle and tip.

In [ ]:
b0 = wl.iloc[0]
BANDS = {
    "name": "Waiting-time bands", "timeUnit": "month", "T": 73, "dt": 0.25,
    "stocks": [stock("b1", "Waiting 0 to 3", b0.w0_3, "patients"),
               stock("b2", "Waiting 3 to 6", b0.w3_6, "patients"),
               stock("b3", "Waiting 6 to 12", b0.w6_12, "patients"),
               stock("b4", "Waiting 12 plus", b0.w12_plus, "patients")],
    "flows": [
        flow("ref", "Referrals", "4000 * (1 + Referral_growth) ^ TIME",
             "patients/month", dst="b1"),
        flow("a1", "Ageing 1", "Waiting_0_to_3 / 3", "patients/month",
             src="b1", dst="b2"),
        flow("a2", "Ageing 2", "Waiting_3_to_6 / 3", "patients/month",
             src="b2", dst="b3"),
        flow("a3", "Ageing 3", "Waiting_6_to_12 / 6", "patients/month",
             src="b3", dst="b4"),
        flow("t1", "Treated 0 to 3", "Treatments * Priority_weight * "
             "Waiting_0_to_3 / Weighted_list", "patients/month", src="b1"),
        flow("t2", "Treated 3 to 6", "Treatments * Waiting_3_to_6 / "
             "Weighted_list", "patients/month", src="b2"),
        flow("t3", "Treated 6 to 12", "Treatments * Waiting_6_to_12 / "
             "Weighted_list", "patients/month", src="b3"),
        flow("t4", "Treated 12 plus", "Treatments * Waiting_12_plus / "
             "Weighted_list", "patients/month", src="b4")],
    "vars": [
        var("g", "Referral growth", 0.0008, "1/month"),
        var("pw", "Priority weight", 1, "dimensionless"),
        var("cap", "Capacity", 3950, "patients/month"),
        var("mtt", "Minimum treatment time", 0.5, "months"),
        var("wlt", "Waiting list", "Waiting_0_to_3 + Waiting_3_to_6 + "
            "Waiting_6_to_12 + Waiting_12_plus", "patients"),
        var("wwl", "Weighted list", "Priority_weight * Waiting_0_to_3 + "
            "Waiting_3_to_6 + Waiting_6_to_12 + Waiting_12_plus",
            "patients"),
        var("trt", "Treatments",
            "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
            "patients/month")]}
bands = lb.Model.from_spec(BANDS)
BAND_COLS = ["w0_3", "w3_6", "w6_12", "w12_plus"]
BAND_NAMES = [s["name"] for s in BANDS["stocks"]]
print(bands.equations())

We fit loosely: a grid of 135 combinations of `Priority weight` and `Capacity`, all run at once with `run_batch`, scored by the root mean square error across the four bands before March 2020.

In [ ]:
#| fig-alt: Four panels, one per band, each with monthly data from 2014 to early 2020 and a dashed model line that follows it closely.
pre = wl[wl.month < "2020-03-01"]
months = np.arange(len(pre), dtype=float)
pw, cap = np.meshgrid(np.arange(0.5, 4.01, 0.25), np.arange(3850, 4051, 25))
grid = pd.DataFrame({"priority_weight": pw.ravel(), "capacity": cap.ravel()})
out = bands.run_batch(grid, T=months[-1])
err = []
for r in grid.index:
    sim = (out[out.run == r].pivot(index="time", columns="variable",
                                   values="value").loc[months, BAND_NAMES])
    err.append(np.sqrt(((sim.values - pre[BAND_COLS].values) ** 2).mean()))
grid["rmse"] = err
best = grid.loc[grid.rmse.idxmin()]
print(grid.sort_values("rmse").head(3).round(1).to_string(index=False))
bands = bands.with_params(priority_weight=best.priority_weight,
                          capacity=best.capacity)

fit = bands.run(T=months[-1])
TITLES = {"w0_3": "0 to 3 months", "w3_6": "3 to 6 months",
          "w6_12": "6 to 12 months", "w12_plus": "12 months and over"}
fig, axes = plt.subplots(2, 2, figsize=(9, 5.5), sharex=True)
for ax, col, name in zip(axes.ravel(), BAND_COLS, BAND_NAMES):
    ax.plot(pre.month, pre[col], color="#000000", lw=1, label="data")
    ax.plot(pre.month, fit[name].loc[months], color="#D55E00", ls="--",
            label="model")
    ax.set(title=TITLES[col], ylabel="patients")
axes[0, 0].legend()
for ax in axes[1]:
    ax.set_xlabel("month")
plt.tight_layout()
plt.show()

Figure: The fitted ageing chain against the four bands, January 2014 to February 2020.

The best pair is a priority weight of 1.5 and a capacity of 3,950 patients a month, with an error of 359.4 patients per band per month. The model follows the rise of the 12-months-and-over band well, but misses the seasonal wiggles and falls about 600 patients short in the youngest band by early 2020: referrals probably grew faster than a constant rate allows. A weight of 1.5 means a patient in the youngest band is half as likely again to be treated in a given month as one who has waited longer. This is loose fitting on purpose: chapter 10 does it with proper uncertainty.

**Try it 3.**

- *Check.* Which band would you expect to grow fastest if capacity falls short of referrals, and why?
- *Apply.* Run the fitted chain to month 72 with priority weights of 1, 1.5 and 3. What share of the list waits 12 months or more? Compare with the data for January 2020.
- *Extend.* Some health systems treat strictly in order of waiting time. How would you represent that in this chain?

<details><summary>Worked answer</summary>

The oldest band: it is the only one with no ageing outflow, so patients pile up there. With weights of 1, 1.5 and 3, the share waiting a year or more is 37.4%, 40.3% and 46.3%; the data say 39.2%. Favouring the newest patients pushes the rest into the long-wait band. Strict order of waiting would put a large weight on the *oldest* band instead, so its treatment share far exceeds its size.
</details>

In [ ]:
for pw_ in (1.0, 1.5, 3.0):
    r = bands.run(T=72, priority_weight=pw_).iloc[-1]
    share = r["Waiting 12 plus"] / r["Waiting list"]
    print(f"priority weight {pw_}: {100 * share:.1f}% wait 12 months or "
          f"more in month 72")
print(f"data, January 2020: "
      f"{100 * wl.w12_plus.iloc[72] / wl.total.iloc[72]:.1f}%")

## 4. Little's law: how long is the wait?

In any system that holds items for a while, the average number inside equals the rate they pass through times the average time each spends there [@little1961]:

$$
L = \lambda\,W \qquad\Longleftrightarrow\qquad W = \frac{L}{\lambda}, \qquad \text{average wait} = \frac{\text{Waiting list}}{\text{Treatments}} .
$$

You met it in chapter 3 as the Residence Time result. It holds for long-run averages whatever the order of treatment, but it needs the flow. Our data have only the stock, so we infer treatments from the books: whatever was referred and did not stay on the list must have been treated. We assume referrals from chapter 3 (4,000 a month, growing by 0.0008 a month).

In [ ]:
t = np.arange(len(wl))
referrals = 4000 * (1 + 0.0008) ** t           # assumed, from chapter 3
change = wl.total.diff().shift(-1)              # list(t + 1) - list(t)
treat = referrals - change                      # what must have left
yearly = pd.DataFrame({"list": wl.total, "treatments": treat,
                       "year": wl.month.dt.year}).groupby("year").mean()
yearly["wait (months)"] = yearly["list"] / yearly["treatments"]
print(yearly.round(1).to_string())

In 2014 the average wait implied by the data is about 8.4 months. It creeps to 10.7 months by 2019, jumps to 21.0 in 2020, when treatments fell to about 2,631 a month on average, and is still 16.1 months in 2023. Two cautions. First, the inferred treatments carry all the error in the assumed referrals.

In [ ]:
r = bands.run(T=72).iloc[-1]
print(f"model, month 72: list {r['Waiting list']:,.0f}, treatments "
      f"{r['Treatments']:,.0f}/month, list / treatments = "
      f"{r['Waiting list'] / r['Treatments']:.2f} months")
for r0 in (3800, 4200):
    tr = r0 * (1 + 0.0008) ** t - change
    w = (wl.total / tr)[wl.month.dt.year == 2023].mean()
    print(f"referrals starting at {r0}/month: 2023 wait {w:.1f} months")

Starting referrals at 3,800 or 4,200 a month moves the 2023 figure between 16.9 and 15.3 months. Second, Little's law is about long-run averages. In 2020, list divided by current treatments describes how long the list would take to clear at that moment's pace, not what any patient experienced. The fitted band model gives 11.21 months in January 2020, close to the data-based figure for 2019.

**Try it 4.**

- *Check.* A clinic has 600 people on its list and sees 150 a month. What is the average wait?
- *Apply.* Using the table above, by how much did the implied wait change between 2014 and 2023? Was it the list or the treatments that drove it?
- *Extend.* Why can't Little's law tell you how long the longest-waiting patients wait? Which part of this chapter can?

<details><summary>Worked answer</summary>

600 / 150 = 4 months. The implied wait rose from 8.4 to 16.1 months. Treatments in 2023 (about 4,330 a month) were higher than in 2014, so the rise came from the list more than doubling. Little's law gives only the average; the spread of waits depends on the order of treatment. The ageing chain of section 3 gives the distribution, band by band.
</details>

## 5. Information delays: perception and smoothing

An **information delay** is a lag in what someone *believes*, not in where things are. Nothing is conserved: a belief can move without anything flowing out of somewhere else. The usual model is **smoothing** (also called adaptive expectations): the belief moves a fraction of the way towards the latest news in each period. The **Smooth** molecule (`smooth`), after Hines, *Molecules of Structure* [@hines_molecules], runs a first-order and a third-order smooth of the same signal:

$$
\frac{dP}{dt} = \frac{X - P}{\tau_s} \quad\text{(first order)}, \qquad \text{third order: three such stages, each with } \tau_s / 3 .
$$

It is the Close Gap molecule with a moving goal. Here $\tau_s$ is the **smoothing time**. Ours runs both orders side by side from the same signal so they can be compared.

| | Material delay | Information delay |
|---|---|---|
| What moves | things (patients, goods, letters) | a belief or a measurement |
| Conserved? | yes: what goes in comes out | no: the old belief is simply replaced |
| Stock means | how much is in transit | the current belief |
| Molecules | Material Delay, Cascaded Levels, Aging Chain | Smooth, Smooth (Higher-Order), Trend |

In [ ]:
#| fig-alt: A signal jumping from 30 to 80, a first-order belief rising steeply then slowly, and a third-order belief that starts slowly and catches up.
sm = lb.molecule("smooth")
print(sm.equations())
sig = sm.run(input_level=80, smoothing_time=6)
fig, ax = plt.subplots()
sig[["Input signal", "First order smooth", "Third order smooth"]].loc[:30] \
    .plot(ax=ax)
ax.set(title="Smooth: a jump from 30 to 80, smoothing time 6 weeks",
       xlabel="time (weeks)", ylabel="signal and belief (units)")
plt.show()
print(sig.loc[[1.0, 3.0, 6.0, 12.0], ["First order smooth",
                                      "Third order smooth"]].round(1))

Figure: Smooth with a smoothing time of 6 weeks. The first-order belief moves at once; the third-order belief waits, then catches up.

The first-order belief reaches 61.8 by week 6; the third-order one hardly moves for two weeks (30.6 in week 1) but overtakes by week 12 (77.1 against 73.3). Official statistics are full of such delays. Germany's statistical office publishes its first estimate of quarterly GDP about a month after the quarter ends, and revises it for years: a decision maker reacting to GDP is reacting to a smoothed, lagged belief.

The **Smooth (Higher-Order)** molecule (`smoothHigher`), after Hines, *Molecules of Structure* [@hines_molecules], chains three first-order smooths with separate times, so you can model people who are slow to notice a change but then catch on fully.

In [ ]:
#| fig-alt: Three S-shaped curves rising from 20 towards 80, reaching the middle at different times.
sh = lb.molecule("smoothHigher")
fig, ax = plt.subplots()
for label in sh.presets:
    run = sh.run(preset=label["label"])
    run["Smooth3"].loc[:25].plot(ax=ax, label=label["label"])
    half = run.index[run["Smooth3"] >= 50][0]
    print(f"{label['label']:<30}: last stage passes 50 in year {half}")
ax.set(title="Smooth (Higher-Order): three chained beliefs",
       xlabel="time (years)", ylabel="last belief (units)")
ax.legend()
plt.show()

Figure: Smooth (Higher-Order) with three choices of stage times. Equal stages give a symmetric S; one long stage gives a quicker start.

With equal stages of 2 years the last belief passes 50 in year 5.4; with one long stage of 5 years and two short ones, in year 4.5, although the total time is the same 6 years; with three stages of 4 years, in year 10.7. Our version differs from the plain Smooth by letting each stage have its own time.

**Try it 5.**

- *Check.* For the jump to 80, when does each belief first reach 70 with a smoothing time of 6 weeks?
- *Apply.* Repeat with 15 weeks.
- *Extend.* Feed a steady 30 with a one-week blip of 10 into the first-order smooth. How high does the belief rise? What would a material delay do with the same blip?

<details><summary>Worked answer</summary>

With 6 weeks: week 9.6 for first order and 8.5 for third order. With 15 weeks: weeks 24.1 and 21.3. Two and a half times the smoothing time means two and a half times as long to notice the same change. The blip lifts the belief only 1.55 above 30, and it is back within 0.1 by week 22.3: smoothing filters noise, and that is why it also hides real change. A material delay would keep all 10 units and deliver every one of them later.

The code makes the blip with `sm.with_equation(name, equation)`, which returns a copy of the model with that one equation replaced and leaves `sm` unchanged; `STEP(10, 5) - STEP(10, 6)` adds 10 from week 5 and takes it away again at week 6. Chapter 5 uses `with_equation` to try out decision rules.
</details>

In [ ]:
for tau in (6, 15):
    run = sm.run(input_level=80, smoothing_time=tau)
    t1 = run.index[run["First order smooth"] >= 70][0]
    t3 = run.index[run["Third order smooth"] >= 70][0]
    print(f"smoothing time {tau:>2}: first order reaches 70 in week {t1}, "
          f"third order in week {t3}")
blip = sm.with_equation("Input signal",
                        "Input_level + STEP(10, 5) - STEP(10, 6)")
x = blip.run(T=30, input_level=30)["First order smooth"]
print(f"a one-week blip of 10 on a steady 30: the belief peaks "
      f"{x.max() - 30:.2f} above 30 and is back within 0.1 by week "
      f"{x.index[(x.index > 6) & (x - 30 < 0.1)][0]}")

Beliefs are often blended from more than one source. The **Weighted Average** molecule (`weightedAvg`), after Hines, *Molecules of Structure* [@hines_molecules], mixes two sources by a weight and lets a stock track the blend:

$$
\text{Blend} = w\,A + (1 - w)\,B, \qquad \frac{dE}{dt} = \frac{\text{Blend} - E}{\tau} .
$$

In [ ]:
wa = lb.molecule("weightedAvg")
print(wa.equations())
for label in ("Favor A", "Even blend"):
    e = wa.run(preset=label)["Estimate"]
    print(f"{label:<10}: estimate {e.loc[4.0]:.1f} in week 4, "
          f"{e.iloc[-1]:.1f} in week 60")

With 80% weight on a source reading 70 and 20% on one reading 30, the estimate moves from 50 towards 62.0, reaching 57.6 by week 4. A GP's sense of the waiting time is like this: part official figures, part what patients say. Our version adds the tracking stock, so the blend is perceived with a delay too.

## 6. Perceived waiting time changes referrals

Now the running case. GPs do not see the true waiting time; they form a belief from published figures and patients' reports. When they believe waits are long, some manage patients in primary care or send them elsewhere, so referrals fall. That is a balancing loop through an information delay:

$$
\text{Referrals} = \text{Base referrals} \times \left(\frac{\text{Perceived wait}}{\text{Reference wait}}\right)^{-s}, \qquad \text{Actual wait} = \frac{\text{Waiting list}}{\text{Treatments}} .
$$

`Perceived wait` is a third-order smooth of `Actual wait` (Little's law at the current pace) with a `Perception time` $\tau_s$; $s$ is the `Referral sensitivity`, 0.3. The model starts in equilibrium, with 32,000 patients and an 8-month wait. Then capacity halves for six months, from month 12 to month 18.

In [ ]:
PERCEIVE = {
    "name": "Perceived waiting time", "timeUnit": "month", "T": 96,
    "dt": 0.125,
    "stocks": [
        stock("wl", "Waiting list", 32000, "patients"),
        stock("p1", "Perceived wait 1", 8, "months"),
        stock("p2", "Perceived wait 2", 8, "months"),
        stock("p3", "Perceived wait", 8, "months")],
    "flows": [
        flow("ref", "Referrals", "Base_referrals * (Perceived_wait / "
             "Reference_wait) ^ (-Referral_sensitivity)", "patients/month",
             dst="wl"),
        flow("trt", "Treatments",
             "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
             "patients/month", src="wl"),
        flow("u1", "Updating 1", "(Actual_wait - Perceived_wait_1) / "
             "Stage_time", "months/month", dst="p1"),
        flow("u2", "Updating 2", "(Perceived_wait_1 - Perceived_wait_2) / "
             "Stage_time", "months/month", dst="p2"),
        flow("u3", "Updating 3", "(Perceived_wait_2 - Perceived_wait) / "
             "Stage_time", "months/month", dst="p3")],
    "vars": [
        var("base", "Base referrals", 4000, "patients/month"),
        var("rw", "Reference wait", 8, "months"),
        var("rs", "Referral sensitivity", 0.3, "dimensionless"),
        var("nc", "Normal capacity", 4000, "patients/month"),
        var("cap", "Capacity", "Normal_capacity * (1 - 0.5 * "
            "(STEP(1, 12) - STEP(1, 18)))", "patients/month"),
        var("mtt", "Minimum treatment time", 0.5, "months"),
        var("aw", "Actual wait", "Waiting_list / Treatments", "months"),
        var("pt", "Perception time", 6, "months"),
        var("st", "Stage time", "Perception_time / 3", "months")]}
perceive = lb.Model.from_spec(PERCEIVE)
print(perceive.equations())

In [ ]:
#| fig-alt: "Left: the waiting list rising during a capacity cut and falling back, peaking higher for longer perception times. Right: referrals falling after the cut, later and less for longer perception times."
runs = {pt: perceive.run(perception_time=pt) for pt in (1, 6, 12)}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.6), sharex=True)
for pt, d in runs.items():
    d["Waiting list"].plot(ax=a1, label=f"perception time {pt}")
    d["Referrals"].plot(ax=a2, label=f"perception time {pt}")
    print(f"perception {pt:>2} months: list peaks at "
          f"{d['Waiting list'].max():,.0f} in month "
          f"{d['Waiting list'].idxmax()}, referrals lowest "
          f"{d['Referrals'].min():,.0f} in month {d['Referrals'].idxmin()},"
          f" lowest list {d['Waiting list'].min():,.0f}")
a1.set(title="Waiting list", xlabel="time (months)",
       ylabel="patients")
a2.set(title="Referrals", xlabel="time (months)",
       ylabel="patients/month")
a1.legend(fontsize=8, title="months", title_fontsize=8)
plt.tight_layout()
plt.show()

Figure: A six-month halving of capacity with three perception times. The slower GPs perceive the change, the higher the list peaks.

With a perception time of 1 month the list peaks at 39,664; with 6 months at 42,559; with 12 months at 43,570. Referrals bottom out later as perception slows: month 18.25, 20.25 and 25.25. By month 96 all three lists are back near 32,000. With 12 months the list even undershoots, to 31,446, because referrals stay low after the backlog has gone: a first glimpse of the swings in chapter 6, Why things swing. Look at what GPs believe during the cut.

In [ ]:
#| fig-alt: Actual waiting time spiking to about 21 months during the cut, and a perceived wait that rises later and lower, peaking near 16 months.
d = runs[6]
fig, ax = plt.subplots()
d["Actual wait"].plot(ax=ax, label="actual wait (list / treatments)")
d["Perceived wait"].plot(ax=ax, label="perceived wait, 6-month smoothing")
ax.set(title="What GPs believe trails what patients face",
       xlabel="time (months)", ylabel="waiting time (months)")
ax.legend()
plt.show()
print(f"actual wait peaks at {d['Actual wait'].max():.1f} months in month "
      f"{d['Actual wait'].idxmax()}; perceived wait peaks at "
      f"{d['Perceived wait'].max():.1f} in month "
      f"{d['Perceived wait'].idxmax()}")

Figure: Actual and perceived waiting time, perception time 6 months.

The actual wait at the current pace peaks at 21.2 months in month 17.875; GPs' belief peaks at only 15.8, more than two months later. Referrals keep falling after capacity has returned, because the belief is still catching up: the information delay makes the correction late and then too long.

**Try it 6.**

- *Check.* Set `referral_sensitivity=0`, so GPs do not react. Where does the list end?
- *Apply.* Compare the list at month 96 with its starting value for each perception time.
- *Extend.* The balancing loop through GPs clears the backlog. What has happened to the patients who were not referred, and why does that matter for a decision maker?

<details><summary>Worked answer</summary>

With no reaction the list ends at 44,000: the 12,000 patients not treated during the cut are never cleared, because capacity only matches referrals. With reaction, the list ends within a few hundred of 32,000 (+374, +130 and -158 for 1, 6 and 12 months). The backlog is cleared by suppressing demand, not by treating it. Those patients are still ill somewhere: in primary care, private care or untreated. A falling list can therefore mean unmet need rather than success, and the model makes that visible.
</details>

In [ ]:
d0 = perceive.run(referral_sensitivity=0)
print(f"no reaction: list at month 96 = {d0['Waiting list'].iloc[-1]:,.0f}")
for pt in (1, 6, 12):
    over = runs[pt]["Waiting list"].iloc[-1] - 32000
    print(f"perception {pt:>2}: list at month 96 is {over:+,.0f} "
          f"against the start")

## 7. Trends and forecasts

Managers plan on forecasts, and forecasts are built from delayed beliefs. The **Trend** molecule (`trend`), after Hines, *Molecules of Structure* [@hines_molecules], smooths a growing quantity; a perception that lags growing reality never catches up. The **Extrapolation** molecule (`extrapolation`), after Hines, *Molecules of Structure* [@hines_molecules], turns the gap between a recent and an older belief into a growth rate and projects forward:

$$
\text{Fractional trend} = \frac{P - H}{H\,\tau_T}, \qquad \text{Forecast} = P \times \big(1 + \text{Fractional trend} \times (\tau_P + h)\big),
$$

where $P$ is the perceived value (smoothed over $\tau_P$), $H$ an older smoothed value (over the trend time $\tau_T$) and $h$ the forecast horizon. Our version uses one-stage smooths for both beliefs.

In [ ]:
tr = lb.molecule("trend")
d = tr.run()
ratio = d["Perceived"] / d["Actual"]
print(f"perceived / actual in month 60: {ratio.iloc[-1]:.3f}; "
      f"theory 1 / (1 + g x tau) = {1 / (1 + 0.06 * 4):.3f}")
ex = lb.molecule("extrapolation")
e = ex.run()
for year in (20, 40):
    print(f"year {year}: forecast for year {year + 5} is "
          f"{e['Extrapolated quantity'][float(year)]:.1f}, actual then "
          f"{e['Actual quantity'][float(year + 5)]:.1f}")

With 6% growth a month and a 4-month perception time, the belief settles at 0.806 of reality, exactly $1/(1 + g\tau)$. The extrapolation corrects most of that lag: its five-year forecast in year 20 is 207.7 against an actual 211.5.

Now apply it to the waiting list. We feed the data into a small forecaster through `inputs` (perception time 3 months, trend time 12 months) and forecast 12 months ahead each month.

In [ ]:
#| fig-alt: "Monthly waiting list with a dashed forecast line made a year earlier: close until 2020, then far below during the jump, then above after it."
FORECAST = {
    "name": "Forecasting the list", "timeUnit": "month", "T": 119,
    "dt": 0.25,
    "stocks": [
        stock("per", "Perceived list", wl.total.iloc[0], "patients"),
        stock("his", "Historical list", wl.total.iloc[0], "patients")],
    "flows": [
        flow("up", "Perceiving", "(Observed_list - Perceived_list) / "
             "Time_to_perceive", "patients/month", dst="per"),
        flow("hi", "Remembering", "(Perceived_list - Historical_list) / "
             "Trend_time", "patients/month", dst="his")],
    "vars": [
        var("obs", "Observed list", 0, "patients"),
        var("tp", "Time to perceive", 3, "months"),
        var("tt", "Trend time", 12, "months"),
        var("h", "Horizon", 12, "months"),
        var("ft", "Fractional trend", "(Perceived_list - Historical_list) /"
            " (Historical_list * Trend_time)", "1/month"),
        var("fc", "Forecast", "Perceived_list * (1 + Fractional_trend * "
            "(Time_to_perceive + Horizon))", "patients")]}
observed = pd.Series(wl.total.values, index=np.arange(len(wl), dtype=float))
f = lb.Model.from_spec(FORECAST).run(inputs={"Observed list": observed})
fc = f["Forecast"].loc[np.arange(108.0)].values      # made in month t
actual = wl.total.values[12:120]                      # month t + 12
err = pd.Series(100 * (fc - actual) / actual,
                index=wl.month.iloc[12:120].values)
for m in ("2016-01", "2018-01", "2019-12", "2020-06", "2021-06"):
    i = wl.index[wl.month == m][0]
    print(f"forecast made {wl.month[i - 12]:%b %Y} for {m}: "
          f"{fc[i - 12]:,.0f} against {wl.total[i]:,}, "
          f"{err.iloc[i - 12]:+.1f}%")

fig, ax = plt.subplots()
ax.plot(wl.month, wl.total, color="#000000", lw=1, label="data")
ax.plot(wl.month.iloc[12:], fc, color="#D55E00", ls="--",
        label="forecast made 12 months earlier")
ax.set(title="Extrapolating the list a year ahead",
       xlabel="month", ylabel="waiting list (patients)")
ax.legend()
plt.show()
print(f"mean absolute error before 2020: {err[:'2019-12'].abs().mean():.1f}%;"
      f" 2020 to 2021: {err['2020-03':'2021-12'].abs().mean():.1f}%")

Figure: The list and the forecast that was made for each month a year before.

Before 2020 the forecast is good, with a mean absolute error of 2.5% (the average miss, as a share of the list; see @sec-learning). The forecast made in June 2019 for June 2020 said 44,469; the list was 55,735, 20.2% higher. Over 2020 and 2021 the mean error is 17.4%. Extrapolation from smoothed history works in steady times and cannot see a turning point: the trend it projects is built from the past.

**Try it 7.**

- *Check.* Why does the Trend molecule's belief settle at a fixed fraction of reality rather than catching up?
- *Apply.* Rerun the forecaster with (perception, trend) times of (1, 6) and (6, 24) months. Which is best before 2020, and which during the shock?
- *Extend.* What information, not in the list itself, would have helped forecast 2020?

<details><summary>Worked answer</summary>

Reality keeps growing during the perception time, so the belief always trails by about one perception time's worth of growth. The errors before 2020 are 2.6%, 2.5% and 3.8%; during 2020 and 2021 they are 19.6%, 17.4% and 18.4%. No setting rescues the forecast: the shock came from capacity, which the list alone cannot reveal. Activity data, such as treatments or theatre sessions, would have shown the cause directly.
</details>

In [ ]:
for tp, tt in ((1, 6), (3, 12), (6, 24)):
    g = lb.Model.from_spec(FORECAST).run(
        inputs={"Observed list": observed}, time_to_perceive=tp,
        trend_time=tt)
    fc2 = g["Forecast"].loc[np.arange(108.0)].values
    e2 = pd.Series(100 * np.abs(fc2 - actual) / actual,
                   index=wl.month.iloc[12:120].values)
    print(f"perceive {tp}, trend {tt:>2}: error before 2020 "
          f"{e2[:'2019-12'].mean():.1f}%, 2020 to 2021 "
          f"{e2['2020-03':'2021-12'].mean():.1f}%")

## 8. Coflows: what patients carry with them

Sometimes the question is not how many are in a stock but what they carry: experience, cost, time waited. A **coflow** tracks such an attribute with a second stock that mirrors the first. The **Coflow** molecule (`coflow`), after Hines, *Molecules of Structure* [@hines_molecules], keeps a stock of total years of experience beside headcount:

$$
\frac{dE}{dt} = \text{Hires} \times e_{\text{hire}} + \text{Headcount} - \text{Attrition} \times \frac{E}{\text{Headcount}} .
$$

The **Coflow with Experience** molecule (`coflowExperience`), after Hines, *Molecules of Structure* [@hines_molecules], tracks the *average* directly; the **Cascaded Coflow** molecule (`cascadedCoflow`), after Hines, *Molecules of Structure* [@hines_molecules], carries an attribute (cost) along a two-stage chain with value added at each stage. Our Coflow starts in balance, and our Coflow with Experience caps the dilution fraction at 5 so that a tiny workforce cannot make the average explode.

In [ ]:
cf = lb.molecule("coflow")
for label in ("Steady team", "Rapid hiring (dilution)", "Hire veterans"):
    per = cf.run(preset=label)["Experience per head"]
    print(f"{label:<24}: experience per head {per.loc[5.0]:.2f} in year 5,"
          f" {per.iloc[-1]:.2f} in year 60")
ce = lb.molecule("coflowExperience")
avg = ce.run(preset="Hiring surge (dilution)")["Average experience"]
print(f"Coflow with Experience, surge: average {avg.min():.2f} at its "
      f"lowest (year {avg.idxmin()}), {avg.iloc[-1]:.2f} in year 60")
cc = lb.molecule("cascadedCoflow")
for label in ("Value added", "No value added"):
    r = cc.run(preset=label).iloc[-1]
    print(f"Cascaded Coflow, {label:<14}: cost per unit "
          f"{r['Avg cost stage 1']:.2f} in stage 1, "
          f"{r['Avg cost stage 2']:.2f} in stage 2")

In the steady team, experience per head stays at 6.00 years, the average tenure. Rapid hiring dilutes it to 4.40 in year 5; hiring veterans with 5 years of experience lifts it to 11.00. The average-based version gives the same lessons: a hiring surge pulls the average down to 6.13, which recovers to 7.98 by year 60. In the cascaded coflow, cost per unit builds from 3.00 in stage 1 to 5.00 in stage 2 with value added, and stays at 1.00 without.

For the running case, the attribute is **time already waited**. Every patient on the list ages by one month each month, and each treated patient takes their time waited with them. If treatment is in random order, a treated patient carries the list's average; a `Selection` below 1 means treated patients have waited less than average:

$$
\frac{d\,TW}{dt} = \text{Waiting list} \times 1 - \text{Treatments} \times \text{Selection} \times \frac{TW}{\text{Waiting list}} .
$$

To compare with the data, we estimate the average time waited from the bands, giving each band its midpoint and assuming 18 months for the top band.

In [ ]:
#| fig-alt: "Average time waited from 2014 to early 2020: the data rise from about 9 to 10 months; a coflow with random selection follows them; one with selection 0.7 rises to about 14."
MID = {"w0_3": 1.5, "w3_6": 4.5, "w6_12": 9.0, "w12_plus": 18.0}
data_avg = sum(wl[c] * m for c, m in MID.items()) / wl.total
WAITED = {
    "name": "Time waited", "timeUnit": "month", "T": 73, "dt": 0.25,
    "stocks": [
        stock("wl", "Waiting list", wl.total.iloc[0], "patients"),
        stock("tw", "Total time waited",
              data_avg.iloc[0] * wl.total.iloc[0], "patient-months")],
    "flows": [
        flow("ref", "Referrals", "4000 * (1 + 0.0008) ^ TIME",
             "patients/month", dst="wl"),
        flow("trt", "Treatments", "MIN(Capacity, Waiting_list / 0.5)",
             "patients/month", src="wl"),
        flow("age", "Ageing", "Waiting_list * 1", "patient-months/month",
             dst="tw"),
        flow("out", "Time treated", "Treatments * Average_time_waited * "
             "Selection", "patient-months/month", src="tw")],
    "vars": [
        var("cap", "Capacity", 3950, "patients/month"),
        var("sel", "Selection", 1, "dimensionless"),
        var("avg", "Average time waited", "Total_time_waited / "
            "Waiting_list", "months")]}
waited = lb.Model.from_spec(WAITED)
fig, ax = plt.subplots()
ax.plot(pre.month, data_avg[:len(pre)], color="#000000", lw=1,
        label="data, band midpoints")
for sel in (1.0, 0.7):
    r = waited.run(selection=sel)["Average time waited"].loc[months]
    ax.plot(pre.month, r, label=f"coflow, selection {sel}")
    print(f"selection {sel}: average time waited {r.iloc[0]:.2f} -> "
          f"{r.iloc[-1]:.2f} months")
print(f"data (band midpoints): {data_avg.iloc[0]:.2f} -> "
      f"{data_avg.iloc[len(pre) - 1]:.2f} months")
ax.set(title="Average time already waited",
       xlabel="month", ylabel="time waited so far (months)")
ax.legend()
plt.show()

Figure: Average time already waited, estimated from the band counts (solid) and from the coflow with two selection rules.

The data estimate rises from 8.84 to 10.09 months. The coflow with random selection rises to 9.98; with selection 0.7 it overshoots to 13.67. On this evidence treatment looks close to random order. But the estimate leans on the assumed 18 months for the top band, as the exercise shows.

**Try it 8.**

- *Check.* In the Coflow molecule, why does the steady team's experience per head equal its average tenure?
- *Apply.* Assume 24 months for the top band. How does the data estimate change, and which selection value now fits?
- *Extend.* Sketch how you would turn the four-band chain of section 3 into a cascaded coflow of time waited. What would you gain?

<details><summary>Worked answer</summary>

In a well-mixed stock, the average time people have been inside equals the average time they stay, so experience per head settles at the tenure of 6 years. With 24 months for the top band, the data estimate rises from 10.61 to 12.45 months, and a selection of about 0.8 (12.17 months) fits better than 1.0 (9.98). The conclusion about treatment order depends on an assumption the data cannot check: say so when you report it. A cascaded coflow would carry time waited within each band, so you could follow the average wait of the longest-waiting group, not only the whole list.
</details>

In [ ]:
mid24 = dict(MID, w12_plus=24.0)
avg24 = sum(wl[c] * m for c, m in mid24.items()) / wl.total
print(f"data with 24 months for the top band: {avg24.iloc[0]:.2f} -> "
      f"{avg24.iloc[len(pre) - 1]:.2f} months")
w24 = waited.with_init("Total time waited", avg24.iloc[0] * wl.total[0])
for sel in (1.0, 0.8, 0.7):
    r = w24.run(selection=sel)["Average time waited"]
    print(f"coflow from that start, selection {sel}: "
          f"{r.loc[months[-1]]:.2f} months in February 2020")

## The molecules of this chapter at a glance

| Molecule | What it decides | Key equation | One run |
|---|---|---|---|
| Material Delay (`material`) | how long goods take in transit | stage outflow = stage / ($D$/3) | 72.0 units in a 9-week pipe of 8 a week |
| Cascaded Levels (`cascade`) | the spread of a delay | chain of first-order stages | a batch's exits peak in month 7.9 |
| Smooth (`smooth`) | how beliefs follow a signal | $dP/dt = (X - P)/\tau_s$ | first order at 61.8 by week 6 |
| Smooth (Higher-Order) (`smoothHigher`) | a slow-to-notice belief | three chained smooths | passes 50 in year 5.4 |
| Trend (`trend`) | lag of a belief behind growth | $P/X \to 1/(1+g\tau)$ | ratio 0.806 |
| Extrapolation (`extrapolation`) | a forecast from a trend | $P(1 + \text{trend}(\tau_P + h))$ | 207.7 against 211.5 |
| Aging Chain (`aging`) | how shocks pass through stages | $S_{i-1}/\tau_{i-1} - S_i/\tau_i$ | elderly 10% down after 33.6 years |
| Aging Chain with PDY (`agingPDY`) | output of a staged workforce | weighted sum of stages | output per person dips to 0.894 |
| Coflow (`coflow`) | an attribute carried by a stock | total attribute / stock | 4.40 years per head during rapid hiring |
| Coflow with Experience (`coflowExperience`) | the average attribute directly | dilution by newcomers | average falls to 6.13 |
| Cascaded Coflow (`cascadedCoflow`) | an attribute along a chain | value added per stage | cost 3.00 then 5.00 per unit |
| Weighted Average (`weightedAvg`) | a blend of two sources | $wA + (1-w)B$, then smoothed | estimate heads for 62.0 |

All are after Hines, *Molecules of Structure* [@hines_molecules]. You can explore any of them interactively in the LoopBricks app, for example the [Smooth molecule](https://waelrash1.github.io/loopbricks/#/m/smooth).

## Check it

For a chain, check that the flows balance across all its stages, that halving `dt` barely moves the slowest stock, and that no floor at zero was needed.

In [ ]:
r = bands.run(T=72)
inflow = r["Referrals"].iloc[:-1].sum() * 0.25
treated = r[[c for c in r.columns if c.startswith("Treated")]]
outflow = treated.iloc[:-1].sum().sum() * 0.25
print(f"referrals - treatments = {inflow - outflow:,.1f}; change in list "
      f"= {r['Waiting list'].iloc[-1] - r['Waiting list'].iloc[0]:,.1f}")
for dt in (0.5, 0.25, 0.125):
    x = bands.run(T=72, dt=dt)["Waiting 12 plus"].iloc[-1]
    print(f"dt = {dt:<5}: 12+ band at month 72 = {x:,.1f}")
print("floors used:", bands.run(T=72).attrs["clipped"] or "none")

Referrals minus all treatments over the run equal the change in the list (12,022.9 patients both ways), the 12-months-and-over band moves by about 30 patients across a fourfold change in `dt`, and no stock hit its floor.

## Running case

This chapter added three pieces to the waiting-list model:

| Element | What it is | Value |
|---|---|---|
| Referral backlog | two-stage material delay before the list | 0.75 months per stage (assumed) |
| Waiting-time bands | ageing chain of four bands, fitted | priority weight 1.5, capacity 3,950 patients/month |
| Perceived wait | third-order smooth of list / treatments, driving referrals | perception time 6 months, sensitivity 0.3 (assumed) |
| Time waited | coflow on the list | selection close to 1 (random order) |

Little's law turns the data into an implied average wait of about 8.4 months in 2014 and 16.1 months in 2023. Chapter 5, Decisions inside the model, adds staffing and treatment rules; chapter 6, Why things swing, shows what happens when a decision acts through these delays.

## Tell the decision maker

- The average wait implied by the counts doubled over the decade, from about 8 to about 16 months, and the main driver was the list more than doubling, not falling activity.
- A falling list after a shock may reflect GPs referring less because they believe waits are long. That is demand suppressed, not demand met. Publish waiting times quickly: the longer GPs take to perceive the wait, the higher the list peaks and the longer referrals stay depressed.
- Forecasts that extrapolate the list's own history were accurate to about 2.5% in steady years and missed by about 17% when capacity changed. Use activity data alongside the list when a change is under way.

## Going further

- **Distributed delays.** Material delays of non-integer order, or with a measured distribution of times, can be built as a convolution of the inflow with a delay kernel; numpy's `convolve` does it in one line.
- **Queueing theory.** Little's law is the bridge to queueing models, which add randomness in arrivals and service; they predict the spread of waits that an ageing chain represents only coarsely.
- **Pipeline order from data.** If entry and exit times are recorded for individual patients, the variance of the waiting time pins down the order: for an $n$-stage delay the coefficient of variation is $1/\sqrt{n}$.

## Mini-project

Make the four-band model react to its own waiting times.

1. Combine section 3 and section 6: let `Referrals` into the youngest band depend on a perceived wait computed from the bands with Little's law.
2. Refit `Priority weight` and `Capacity` to the bands before March 2020 with the reaction switched on and off. Does adding the loop change the fitted values?
3. Hit the fitted model with the capacity cut of section 6 and report, for a decision maker, the extra patients waiting over 12 months at the worst point, with and without a 6-month perception time.

## Further reading

- @sterman2000, chapters 11 and 12, treats delays, ageing chains and coflows in depth.
- @little1961 gives the original proof of the queueing formula used in section 4.
- @hines_molecules is the source of the molecules used here.
- @forrester1961 introduced the third-order delay and the treatment of information as a flow distinct from materials.